[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C12_Responsible_AI_Course/04_memorization_privacy/04_memorization_privacy.ipynb)

# 04 · 记忆化与隐私（用 numpy 从零实现）

目标：把 **canary 暴露度（exposure）**、**成员推断攻击（MIA）的 loss 阈值法 + ROC/AUC**、**重复次数→记忆**、**去重缓解** 全部用纯 numpy/pandas 实现，并用 `assert` 验证。

路线：toy 语言模型(计数 n-gram) → canary 排名与暴露度 → 暴露度随重复上升 → MIA 损失分布与 ROC/AUC → 低 FPR 区为何重要(LiRA 直觉) → 去重缓解 → ✏️ 练习 → 📖 答案 → 🧪 真实文本胶囊。

> 心智模型：**记忆 = 模型对见过的数据给出反常的低损失**。canary 暴露、提取、成员推断，底层都是这一个信号。

## 1 · 一个玩具 “语言模型”：计数 n-gram 打分器

我们不需要真模型。用一个 **bigram 计数模型** 当 toy LM：在训练文本上数 `(前一字符 → 当前字符)` 的频次，用平滑后的条件概率给任意字符串打 **负对数似然损失**(loss)——一段文本若被模型 “熟悉”，loss 就低。

记忆将被这样模拟：**把某段文本在训练语料里重复多次**，它的 bigram 统计被反复加强，模型对它的 loss 就异常低——这正是记忆的本质。

In [ ]:
import numpy as np
import pandas as pd
rng = np.random.default_rng(0)

class CountLM:
    '''玩具 bigram 计数语言模型：loss(s) = 平均负对数条件概率。'''
    def __init__(self, alpha=1.0):
        self.alpha = alpha            # 加性平滑
        self.bigram = {}              # (a,b) -> count
        self.unigram = {}             # a -> count
        self.vocab = set()
    def train(self, texts):
        for t in texts:
            t = '^' + t               # 开头符
            for a, b in zip(t, t[1:]):
                self.bigram[(a, b)] = self.bigram.get((a, b), 0) + 1
                self.unigram[a] = self.unigram.get(a, 0) + 1
                self.vocab.add(a); self.vocab.add(b)
        return self
    def loss(self, s):
        '''平均负对数似然(越低=模型越熟悉这段文本)。'''
        s = '^' + s
        V = max(len(self.vocab), 1)
        total = 0.0
        for a, b in zip(s, s[1:]):
            num = self.bigram.get((a, b), 0) + self.alpha
            den = self.unigram.get(a, 0) + self.alpha * V
            total += -np.log(num / den)
        return total / max(len(s) - 1, 1)

# 训练语料：普通英文短句 + 一些含【随机】数字的 “secret code” 模板句。
# 后者让数字 bigram 进入词表(否则数字全是 OOV, canary 无法与候选区分)，
# 但因为数字是随机的, 干净模型不会偏爱任何特定 6 位串 —— 记忆必须靠重复插入制造。
_r0 = np.random.default_rng(123)
_code_lines = ['the secret code is ' + ' '.join(_r0.choice(list('0123456789'), size=6))
               for _ in range(40)]
corpus = (['the cat sat on the mat', 'a dog ran in the park',
           'she sells sea shells', 'the quick brown fox jumps',
           'rain in spain stays mainly'] * 4) + _code_lines
lm = CountLM().train(corpus)
print('词表大小:', len(lm.vocab))
l_seen   = lm.loss('the cat sat on the mat')     # 训练里出现
l_unseen = lm.loss('xqzj vbnm wkpl')             # 完全陌生
print(f'熟悉句 loss = {l_seen:.3f}   陌生串 loss = {l_unseen:.3f}')
assert l_seen < l_unseen, '见过的文本损失应更低'
print('✅ toy LM 就绪：见过的文本 loss 更低 —— 这就是记忆的可测量信号')

## 2 · canary 与暴露度：往训练集埋一条随机串

插入一条 **canary**(随机唯一串，如 `the secret code is 8 2 7 3 5 1`)。它的填空槽是 6 位数字，候选空间 $|R|=10^6$。

**暴露度** 衡量模型对真 canary 的偏爱程度：把真 canary 与一批同格式随机候选按 loss 排序，真 canary 的名次记作 `rank`，

$$\mathrm{exposure} = \log_2 |R| - \log_2 \mathrm{rank}$$

没记住 → rank 在中间 → exposure ≈ 1；完全记住 → rank=1 → exposure = $\log_2|R|$(最大)。

In [ ]:
DIGITS = '0123456789'
R_SIZE = 10**6            # 6 位数字的候选空间大小

def make_canary(code):
    digs = ' '.join(list(code))
    return f'the secret code is {digs}'

def random_candidates(n, seed=1):
    r = np.random.default_rng(seed)
    return [''.join(r.choice(list(DIGITS), size=6)) for _ in range(n)]

def exposure(lm, secret_code, n_candidates=2000, seed=1):
    '''用一批随机候选估计真 canary 的 rank，换算 exposure。
       frac = 候选中比真 canary 更被偏好(loss 更低)的比例; 平局算一半,
       使未记住时 frac≈0.5 → rank≈|R|/2 → exposure≈1(纯属偶然)。'''
    true_loss = lm.loss(make_canary(secret_code))
    cand = random_candidates(n_candidates, seed)
    cand_losses = np.array([lm.loss(make_canary(co)) for co in cand])
    # loss 严格更低算 “更被偏好”; 平局(数值相同)各算一半
    lower = float(np.mean(cand_losses < true_loss - 1e-12))
    tie   = float(np.mean(np.abs(cand_losses - true_loss) <= 1e-12))
    frac_better = lower + 0.5 * tie
    # 估计在完整候选空间 R 中的 rank(至少为 1)
    est_rank = max(frac_better * R_SIZE, 1.0)
    return np.log2(R_SIZE) - np.log2(est_rank), est_rank, true_loss

SECRET = '827351'
# 情形 A: canary 没插入训练集(模型没见过)
lm_clean = CountLM().train(corpus)
exp_clean, rank_clean, _ = exposure(lm_clean, SECRET)
# 情形 B: canary 插入训练集 50 次(模型反复见过)
lm_mem = CountLM().train(corpus + [make_canary(SECRET)] * 50)
exp_mem, rank_mem, _ = exposure(lm_mem, SECRET)
print(f'未插入 : exposure={exp_clean:5.2f}  估计 rank≈{rank_clean:,.0f}')
print(f'插入50次: exposure={exp_mem:5.2f}  估计 rank≈{rank_mem:,.0f}')
print(f'最大可能 exposure = log2(1e6) = {np.log2(R_SIZE):.2f}')
assert exp_mem > exp_clean + 3, '插入后暴露度应显著升高'
print('✅ 被记住的 canary 暴露度飙升 —— Secret Sharer 的核心信号')

## 3 · 暴露度随重复次数上升

Carlini 反复观测到的规律：一段文本在训练集里 **出现越多次，被记得越牢**。我们扫描插入次数，看暴露度单调上升——这正是后面 “去重为何有效” 的根据(把高频重复砍成低频，记忆就掉下来)。

In [ ]:
reps_list = [0, 1, 3, 10, 30, 100]
rows = []
for reps in reps_list:
    lm_r = CountLM().train(corpus + [make_canary(SECRET)] * reps)
    exp_r, rank_r, _ = exposure(lm_r, SECRET)
    rows.append(dict(repetitions=reps, exposure=round(exp_r, 2), est_rank=int(rank_r)))
tab = pd.DataFrame(rows)
print(tab.to_string(index=False))
# 暴露度应随重复次数(弱)单调上升
exps = tab['exposure'].values
assert exps[-1] > exps[0] + 3, '重复越多暴露度越高'
assert exps[-1] >= exps[1], '更多重复不应降低暴露度'
print('✅ 重复次数↑ → 暴露度↑：记忆强烈依赖重复，这是去重缓解的依据')

## 4 · 成员推断(MIA)：损失分布、ROC 与 AUC

**MIA**：判断一条样本是否在训练集中。最简单的 **loss 阈值攻击**——成员(见过)的 loss 偏低，非成员偏高，于是 “loss < τ 判为成员”。扫遍 τ 画 ROC，算 **AUC**。

AUC 用 Mann–Whitney 等价定义从零实现：**随机抽一个成员、一个非成员，成员 loss 更低的概率**。AUC=0.5 表示无泄露。

In [ ]:
def auc_from_scores(member_score, nonmember_score):
    '''AUC = P(随机成员比随机非成员更像成员)。这里 “更像成员” = loss 更低,
       所以用 (-loss) 当 “成员性分数”, 数值越大越像成员。rank 法实现, 处理并列。'''
    s_in = -np.asarray(member_score)       # 成员性分数: loss 越低分越高
    s_out = -np.asarray(nonmember_score)
    all_s = np.concatenate([s_in, s_out])
    order = all_s.argsort()
    ranks = np.empty(len(all_s)); ranks[order] = np.arange(1, len(all_s) + 1)
    # 处理并列: 同值取平均秩
    uniq, inv, counts = np.unique(all_s, return_inverse=True, return_counts=True)
    avg_rank = np.zeros(len(uniq))
    cum = np.cumsum(counts)
    start = cum - counts
    for k in range(len(uniq)):
        avg_rank[k] = (start[k] + 1 + cum[k]) / 2.0
    ranks = avg_rank[inv]
    n_in = len(s_in); n_out = len(s_out)
    sum_in = ranks[:n_in].sum()
    auc = (sum_in - n_in * (n_in + 1) / 2.0) / (n_in * n_out)
    return float(auc)

def roc_curve_losses(member_loss, nonmember_loss, n_thresh=200):
    '''扫阈值: loss<τ 判成员。返回 (fpr, tpr) 数组。'''
    lo = min(member_loss.min(), nonmember_loss.min())
    hi = max(member_loss.max(), nonmember_loss.max())
    taus = np.linspace(lo - 1e-6, hi + 1e-6, n_thresh)
    tpr = np.array([np.mean(member_loss < t) for t in taus])
    fpr = np.array([np.mean(nonmember_loss < t) for t in taus])
    return fpr, tpr

# 构造成员/非成员: 用较丰富的字母串当 “记录”(像 PII), 成员被训练(重复→记忆其特有 bigram),
# 非成员是同样方式生成但【未参与训练】的新记录。成员 loss 应更低 → MIA 可区分。
def gen_record(r):
    '''生成一条像 “user kxqv lives at zmph town” 的随机记录(含罕见字母组合)。'''
    def token():
        return ''.join(r.choice(list('abcdefghijklmnopqrstuvwxyz'), size=r.integers(4, 8)))
    return 'user ' + token() + ' lives at ' + token() + ' ' + token()

r = np.random.default_rng(7)
members    = [gen_record(r) for _ in range(150)]
nonmembers = [gen_record(r) for _ in range(150)]
# 成员重复多次进入训练 → 模型记住它们特有的字符 bigram
lm_mia = CountLM().train(corpus + members * 8)
ml = np.array([lm_mia.loss(s) for s in members])
nl = np.array([lm_mia.loss(s) for s in nonmembers])
print(f'成员平均 loss = {ml.mean():.3f}   非成员平均 loss = {nl.mean():.3f}')
auc = auc_from_scores(ml, nl)
print(f'MIA AUC = {auc:.3f}  (0.5=无泄露, 越高泄露越严重)')
fpr, tpr = roc_curve_losses(ml, nl)
assert ml.mean() < nl.mean(), '成员损失应更低'
assert auc > 0.6, '成员被训练后 MIA 应显著优于随机'
assert fpr[0] <= fpr[-1] and tpr[0] <= tpr[-1], 'ROC 应单调'
print('✅ loss 阈值 MIA：成员损失更低 → AUC>0.5 → 隐私泄露可量化')

## 5 · 为什么要看 **低 FPR 区**(LiRA 的核心)

Carlini 2022(LiRA)指出：**平均 AUC 会骗你**。隐私伤害是个体化的——攻击者只要能 **高置信确认少数人** 在训练集里就够灾难，哪怕平均 AUC 温和。正确度量是看 ROC **低假阳率区**(如 FPR=1%)的 TPR：“几乎不冤枉无辜时，能抓出多少真成员”。

我们实现 `tpr_at_fpr`，并构造一小撮 “异常易记” 的样本，看它们如何在极低 FPR 处被高置信识别。

In [ ]:
def tpr_at_fpr(member_loss, nonmember_loss, target_fpr=0.01):
    '''在给定 FPR 工作点上的 TPR: 阈值取 “非成员 loss 的 target_fpr 分位数”。'''
    tau = np.quantile(nonmember_loss, target_fpr)   # 只允许 target_fpr 比例的非成员被误判
    tpr = float(np.mean(member_loss < tau))
    return tpr, float(tau)

# 一般成员 + 一小撮 “异常易记”样本(被重复很多次, loss 极低)
outliers = ['zzz qqq vvv www', 'kkk jjj fff ggg']        # 罕见 → 易区分
lm_out = CountLM().train(corpus + members * 2 + outliers * 80)
ml2 = np.array([lm_out.loss(s) for s in members])
nl2 = np.array([lm_out.loss(s) for s in nonmembers])
ol2 = np.array([lm_out.loss(s) for s in outliers])
auc_all = auc_from_scores(np.concatenate([ml2, ol2]), nl2)
tpr_lo, tau = tpr_at_fpr(np.concatenate([ml2, ol2]), nl2, target_fpr=0.01)
# 单看异常样本: 它们的 loss 是否低于这个低-FPR 阈值(即被高置信抓出)
outlier_caught = float(np.mean(ol2 < tau))
print(f'整体 AUC = {auc_all:.3f}')
print(f'FPR=1% 处 TPR = {tpr_lo:.3f}')
print(f'异常易记样本在 FPR=1% 阈值下被抓出的比例 = {outlier_caught:.2f}')
assert outlier_caught >= 0.5, '异常易记样本应在极低 FPR 处就被高置信识别'
print('✅ 平均数掩盖尾部: 对异常样本, 隐私在极低 FPR 处已被击穿 —— 必须看低 FPR 区')

## 6 · 去重缓解：把重复砍掉, 记忆就掉下来

既然记忆强烈依赖重复(第 3 节), **去重**(deduplication)就是最划算的缓解：训练前删掉重复样本, 高频 canary 变低频, 暴露度随之下降。下面对比 “去重前(canary 重复 60 次)” vs “去重后(只剩 1 次)” 的暴露度。

In [ ]:
def dedup(texts):
    '''最简去重: 保留每个唯一文本一次(保持顺序)。'''
    seen = set(); out = []
    for t in texts:
        if t not in seen:
            seen.add(t); out.append(t)
    return out

raw_corpus = corpus + [make_canary(SECRET)] * 60         # 含高频重复的 canary
lm_before = CountLM().train(raw_corpus)
lm_after  = CountLM().train(dedup(raw_corpus))           # 去重后 canary 只剩 1 次
exp_before, _, _ = exposure(lm_before, SECRET)
exp_after,  _, _ = exposure(lm_after,  SECRET)
print(f'去重前(重复60次) exposure = {exp_before:.2f}')
print(f'去重后(仅 1 次)  exposure = {exp_after:.2f}')
print(f'去重把暴露度降低了 {exp_before - exp_after:.2f}')
assert exp_after < exp_before - 2, '去重应显著降低暴露度'
print('✅ 去重: 一次数据清洗就大幅压低记忆 —— 性价比最高的隐私缓解')

---
## ✏️ 练习 1：从零实现暴露度

给定真 canary 的 loss 和一组随机候选的 loss 数组, 实现暴露度。

步骤: ① 算 “候选中 loss 比真 canary 更低的比例” `frac` ; ② 估计 rank = `max(frac * R_size, 1)` ; ③ 返回 `log2(R_size) - log2(rank)`。

In [ ]:
def exposure_from_losses(true_loss, candidate_losses, R_size):
    # TODO: 见上面三步。candidate_losses 是 np.array
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
cand = np.array([5.0, 4.8, 6.1, 5.5, 4.9])      # 5 个候选
# 真 canary loss 比所有候选都低 → 应排第 1 → 暴露度最大
e_best = exposure_from_losses(3.0, cand, 1000)
assert abs(e_best - np.log2(1000)) < 1e-9, '真 canary 最优时 exposure=log2(R)'
# 真 canary loss 在中间(一半候选更低) → rank≈R/2 → exposure≈1
e_mid = exposure_from_losses(5.05, np.array([4.0,4.5,6.0,6.5]), 1000)
assert abs(e_mid - 1.0) < 1e-9, '真 canary 居中时 exposure≈1'
print('✅ 练习 1 通过：暴露度实现正确')

## ✏️ 练习 2：MIA 的 TPR@FPR

实现 `mia_tpr_at_fpr(member_loss, nonmember_loss, target_fpr)`: 在只允许 `target_fpr` 比例非成员被误判的工作点上, 返回 (TPR, 阈值)。提示: 阈值 = 非成员 loss 的 `target_fpr` 分位数, TPR = 成员中 loss < 阈值的比例。

In [ ]:
def mia_tpr_at_fpr(member_loss, nonmember_loss, target_fpr=0.01):
    # TODO: tau = np.quantile(nonmember_loss, target_fpr); tpr = mean(member_loss < tau)
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
rr = np.random.default_rng(3)
mem = rr.normal(1.0, 0.5, 500)        # 成员 loss 低
non = rr.normal(3.0, 0.5, 500)        # 非成员 loss 高
tpr, tau = mia_tpr_at_fpr(mem, non, target_fpr=0.05)
assert 0.0 <= tpr <= 1.0
assert tpr > 0.9, '两分布分得很开时, 低 FPR 处 TPR 应很高'
# 分布完全重叠时 TPR≈target_fpr(无信息)
same = rr.normal(2.0, 0.5, 2000)
tpr2, _ = mia_tpr_at_fpr(same[:1000], same[1000:], target_fpr=0.1)
assert abs(tpr2 - 0.1) < 0.06, '无泄露时 TPR≈FPR'
print(f'分得开: TPR@FPR5%={tpr:.2f} | 重叠: TPR@FPR10%={tpr2:.2f}')
print('✅ 练习 2 通过：低 FPR 工作点的 TPR 正确')

## ✏️ 练习 3：去重对暴露度的影响

实现 `exposure_after_dedup(corpus, secret_code, reps)`: 把 canary 重复 `reps` 次混入 corpus, **去重后** 训练 toy LM, 返回该 canary 的暴露度。复用上面的 `dedup`、`CountLM`、`make_canary`、`exposure`。验证: 去重后暴露度 ≈ 只插一次的水平(远低于不去重)。

In [ ]:
def exposure_after_dedup(corpus, secret_code, reps):
    # TODO: raw = corpus + [make_canary(secret_code)]*reps; 去重; 训练; 返回 exposure(...)[0]
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
exp_dedup = exposure_after_dedup(corpus, '135790', reps=80)
# 对照: 不去重直接重复 80 次
lm_nodedup = CountLM().train(corpus + [make_canary('135790')] * 80)
exp_nodedup, _, _ = exposure(lm_nodedup, '135790')
assert exp_dedup < exp_nodedup - 2, '去重应显著降低暴露度'
print(f'去重后 exposure={exp_dedup:.2f}  不去重 exposure={exp_nodedup:.2f}')
print('✅ 练习 3 通过：去重把高频 canary 的记忆压了下去')

## ✏️ 练习 4：隐私风险评估与排序

给一个 DataFrame, 每行是一条 canary 的 `name` 与 `exposure`。实现 `risk_report(df, exposure_thresh)`: 
① 加一列 `high_risk` = exposure ≥ 阈值 ; ② 按 exposure 降序排序 ; ③ 返回排序后的 DataFrame。这模拟把记忆度量转成 “先处理哪条” 的优先级。

In [ ]:
def risk_report(df, exposure_thresh=10.0):
    # TODO: df=df.copy(); df['high_risk']=df['exposure']>=thresh; 按 exposure 降序; return
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
canaries = pd.DataFrame({
    'name': ['ssn_canary', 'phone_canary', 'random_str', 'address_canary'],
    'exposure': [15.2, 3.1, 1.0, 11.8],
})
rep = risk_report(canaries, exposure_thresh=10.0)
assert list(rep['name'])[0] == 'ssn_canary', '暴露度最高的应排最前'
assert rep['high_risk'].sum() == 2, '应有 2 条高风险(≥10)'
assert list(rep['exposure']) == sorted(rep['exposure'], reverse=True), '应按暴露度降序'
print(rep.to_string(index=False))
print('✅ 练习 4 通过：能把记忆度量转成风险优先级')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def exposure_from_losses(true_loss, candidate_losses, R_size):
    frac = float(np.mean(np.asarray(candidate_losses) < true_loss))
    rank = max(frac * R_size, 1.0)
    return np.log2(R_size) - np.log2(rank)

In [ ]:
# 练习 2 参考答案
def mia_tpr_at_fpr(member_loss, nonmember_loss, target_fpr=0.01):
    tau = np.quantile(nonmember_loss, target_fpr)
    tpr = float(np.mean(np.asarray(member_loss) < tau))
    return tpr, float(tau)

In [ ]:
# 练习 3 参考答案
def exposure_after_dedup(corpus, secret_code, reps):
    raw = corpus + [make_canary(secret_code)] * reps
    lm = CountLM().train(dedup(raw))
    return exposure(lm, secret_code)[0]

In [ ]:
# 练习 4 参考答案
def risk_report(df, exposure_thresh=10.0):
    df = df.copy()
    df['high_risk'] = df['exposure'] >= exposure_thresh
    return df.sort_values('exposure', ascending=False).reset_index(drop=True)

---
## 🧪 真实数据胶囊：在真实文本上测 canary 暴露度

下面尝试联网下载一本 **公有领域真实文本**(Project Gutenberg 的《爱丽丝梦游仙境》)当训练语料, 在 **真实英文统计** 上训练 toy LM, 再插入 canary 测暴露度——看记忆信号在真实文本(而非玩具句子)上同样成立。

**下载失败会自动回退到内置的真实英文片段**(取自该书开头), 逻辑与结论不变: 被重复插入的 canary 暴露度远高于未插入。

In [ ]:
import urllib.request

ALICE_URL = 'https://www.gutenberg.org/files/11/11-0.txt'
FALLBACK_TEXT = (
    'alice was beginning to get very tired of sitting by her sister on the bank '
    'and of having nothing to do once or twice she had peeped into the book her '
    'sister was reading but it had no pictures or conversations in it and what is '
    'the use of a book thought alice without pictures or conversations so she was '
    'considering in her own mind whether the pleasure of making a daisy chain would '
    'be worth the trouble of getting up and picking the daisies when suddenly a '
    'white rabbit with pink eyes ran close by her ')

def load_real_text():
    '''返回 (sentences, source)。联网失败回退内置真实片段。'''
    try:
        req = urllib.request.Request(ALICE_URL, headers={'User-Agent': 'Mozilla/5.0'})
        raw = urllib.request.urlopen(req, timeout=8).read().decode('utf-8', 'replace')
        # 取正文中间一段, 规范化为小写字母与空格
        body = raw[5000:30000].lower()
        import re
        body = re.sub(r'[^a-z ]+', ' ', body)
        words = body.split()
        # 切成每 8 词一句
        sents = [' '.join(words[i:i+8]) for i in range(0, len(words) - 8, 8)][:400]
        return sents, 'Project Gutenberg: Alice (downloaded)'
    except Exception as e:
        print('下载失败, 回退内置真实片段:', type(e).__name__)
        words = FALLBACK_TEXT.split()
        sents = [' '.join(words[i:i+8]) for i in range(0, len(words) - 8, 4)]
        return sents * 6, 'built-in (real excerpt of Alice)'

real_sents, src = load_real_text()
print('数据来源:', src, '| 句子数:', len(real_sents))
lm_real_clean = CountLM().train(real_sents)
lm_real_mem   = CountLM().train(real_sents + [make_canary('482913')] * 40)
exp_real_clean, _, _ = exposure(lm_real_clean, '482913')
exp_real_mem,   _, _ = exposure(lm_real_mem,   '482913')
print(f'真实文本上 — 未插入 exposure={exp_real_clean:.2f} | 插入40次 exposure={exp_real_mem:.2f}')
assert exp_real_mem > exp_real_clean + 3, '真实文本上插入的 canary 同样高暴露'
print('✅ 记忆信号在真实文本上同样成立 —— canary 暴露度是通用的隐私探针')

**🧪 胶囊练习**：实现 `is_memorized(lm, secret_code, exposure_thresh)`: 算该 canary 在 `lm` 上的暴露度, 返回 `(是否被记住的布尔, 暴露度)`。用它判定上面 `lm_real_mem` 记住了 canary、`lm_real_clean` 没有。

In [ ]:
def is_memorized(lm, secret_code, exposure_thresh=5.0):
    # TODO: 用 exposure(lm, secret_code)[0] 取暴露度, 与阈值比较
    raise NotImplementedError

In [ ]:
# 自测
mem_flag, e_mem = is_memorized(lm_real_mem, '482913', exposure_thresh=5.0)
clean_flag, e_clean = is_memorized(lm_real_clean, '482913', exposure_thresh=5.0)
assert mem_flag is True or mem_flag == True, '插入版应判为已记住'
assert not clean_flag, '干净版不应判为已记住'
print(f'插入版: memorized={mem_flag}(exp={e_mem:.2f}) | 干净版: memorized={clean_flag}(exp={e_clean:.2f})')
print('✅ 胶囊练习通过：能据暴露度判定一条数据是否被记住')

In [ ]:
# 📖 胶囊参考答案
def is_memorized(lm, secret_code, exposure_thresh=5.0):
    e = exposure(lm, secret_code)[0]
    return bool(e >= exposure_thresh), e

### 小结
- **记忆 = 见过的数据 loss 反常地低**——canary 暴露、提取、成员推断都是这一个信号的不同用法。
- **暴露度** `= log2|R| − log2(rank)`：把 “模型多偏爱这条 canary” 变成可比的数；重复越多, 暴露度越高。
- **MIA(loss 阈值)** 把隐私泄露变成一张 ROC；但 **平均 AUC 会骗你**——隐私伤害个体化, 必须看 **低 FPR 区的 TPR**(LiRA)。
- **去重** 是性价比最高的缓解(掐断高频重复→记忆掉下来)；**差分隐私** 最彻底但牺牲效用。报告 **残余风险**, 不是 “我们做了缓解”。

下一站：**模块 05 · 危害分类学与社会影响** —— 把公平/毒性/多语/隐私这些零散危害, 汇成一张能排优先级的账本。